# Notebook 6 – Outlier Treatment


An outlier is an observation unusually far from the bulk of the data. It may be an error, a rare but valid event, or a signal of an important business condition. Statistical detection alone does not tell us which.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Example data
The values include a deliberately extreme observation. We will inspect it rather than deleting it automatically.

In [ ]:
sales = pd.Series([95, 100, 102, 98, 101, 97, 103, 99, 105, 100, 500],
                  name="daily_sales")
sales.describe()

In [ ]:
plt.figure(figsize=(7, 3))
plt.boxplot(sales, vert=False)
plt.title("Daily sales: box plot")
plt.xlabel("Sales")
plt.show()

## 2. IQR method
The interquartile range (IQR) is Q3 − Q1. A common rule flags values below Q1 − 1.5×IQR or above Q3 + 1.5×IQR. These are screening thresholds, not automatic deletion instructions.

In [ ]:
q1 = sales.quantile(0.25)
q3 = sales.quantile(0.75)
iqr = q3 - q1
lower_iqr = q1 - 1.5 * iqr
upper_iqr = q3 + 1.5 * iqr
iqr_outliers = sales[(sales < lower_iqr) | (sales > upper_iqr)]
print({"Q1": q1, "Q3": q3, "IQR": iqr,
       "lower_bound": lower_iqr, "upper_bound": upper_iqr})
iqr_outliers

## 3. Z-score method
A Z-score measures how many standard deviations an observation is from the mean. A common screening threshold is an absolute Z-score above 3, but this method can be sensitive to skewed data and extreme values.

In [ ]:
mean = sales.mean()
std = sales.std(ddof=0)
z_scores = (sales - mean) / std
z_outliers = sales[z_scores.abs() > 3]
pd.DataFrame({"value": sales, "z_score": z_scores,
              "flagged_by_zscore": z_scores.abs() > 3})

## 4. Percentile method
Percentile cutoffs can be useful when business policy defines the extreme tail (for example, investigate the top 1%). Percentiles always flag a chosen proportion; that does not prove the values are wrong.

In [ ]:
lower_p = sales.quantile(0.01)
upper_p = sales.quantile(0.99)
percentile_flags = sales[(sales < lower_p) | (sales > upper_p)]
print("1st percentile:", lower_p, "| 99th percentile:", upper_p)
percentile_flags

## 5. Treatment options
- **Retain:** keep a valid extreme value, especially when it is meaningful to the problem.
- **Remove:** only when it is confirmed invalid or outside the population being studied; record the reason.
- **Capping / Winsorization:** replace values beyond selected limits with the boundary values. This reduces extreme influence but changes the data.
- **Flooring:** set values below a lower threshold to that threshold.
- **Transformation:** log or another suitable transformation can reduce right skew; it does not fix data errors and requires nonnegative/positive-domain care.

In [ ]:
# Capping / Winsorization with IQR bounds
capped = sales.clip(lower=lower_iqr, upper=upper_iqr)

# Example log1p transformation for nonnegative data
log_transformed = np.log1p(sales)

comparison = pd.DataFrame({
    "original": sales,
    "capped": capped,
    "log1p_transformed": log_transformed
})
comparison

## 6. Investigate each detected outlier
For this example, `500` is statistically unusual. It might be a data-entry error (for example, an extra zero), or it could be a legitimate promotion/large order day.

| Question | Decision process |
|---|---|
| Is it an error? | Check the source record, units, and ingestion logs. |
| Is it a valid extreme observation? | Compare with invoices, campaign dates, and business context. |
| Should it be removed? | Only if confirmed invalid or excluded by the study definition. |
| Should it be capped? | Consider if extreme values distort a model and capping is defensible. |
| Should it be transformed? | Consider for strongly skewed positive data and models that benefit from it. |
| Why? | Document evidence, business impact, and model/evaluation consequences. |

In [ ]:
# Create a review table instead of automatically deleting flagged values
review = pd.DataFrame({"value": sales, "flagged_iqr": (sales < lower_iqr) | (sales > upper_iqr)})
review["initial_action"] = np.where(review["flagged_iqr"], "Investigate", "No outlier flag")
review